# Structured Output Configuration Tutorial

This tutorial demonstrates how to configure agents with structured output schemas using YAML configuration files.

## What You'll Learn
- Define structured output schemas in YAML configuration
- Use inline schemas, external files, and existing Pydantic models
- Configure validation and error handling
- Build a complete business intelligence pipeline

## Prerequisites
- Basic understanding of Pydantic models
- Familiarity with YAML configuration
- AWS credentials configured for Bedrock access

In [ ]:
!pip install strands-agents strands-agents-tools python-dotenv PyYAML

In [ ]:
import yaml
import json
from pathlib import Path
from datetime import datetime
from IPython.display import display, Markdown, JSON

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

# Import Strands components
from strands.experimental.config_loader.agent import AgentConfigLoader

print("✅ Environment setup complete")

In [ ]:
# Simple structured output configuration
simple_config = {
    "schemas": [
        {
            "name": "UserProfile",
            "schema": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "description": "User's full name"},
                    "email": {"type": "string"},
                    "age": {"type": "integer", "minimum": 0}
                },
                "required": ["name", "email"]
            }
        }
    ],
    "agent": {
        "name": "user_extractor",
        "model": "us.anthropic.claude-3-7-sonnet-20250219-v1:0",
        "system_prompt": "Extract user information from text.",
        "structured_output": "UserProfile"
    }
}

# Load agent
loader = AgentConfigLoader()
agent = loader.load_agent(simple_config)

print("✅ Loaded agent with structured output")
print(f"📋 Schema: {agent._structured_output_schema.__name__}")

In [ ]:
# Demo structured output extraction
sample_text = "Hi, I'm John Doe, 30 years old. You can reach me at john.doe@example.com"

print("📄 Sample Text:")
print(sample_text)

print("\n🔍 Extracting structured data...")
result = agent.structured_output(f"Extract user information: {sample_text}")

print("\n✅ Extraction completed!")
print(f"Name: {result.name}")
print(f"Email: {result.email}")
print(f"Age: {result.age}")

# Show full structured data
display(JSON(result.model_dump(), expanded=True))

## Advanced Configuration with Validation

Now let's explore more advanced structured output configuration with validation and error handling.

In [ ]:
# Advanced configuration with validation
advanced_config = {
    "schemas": [
        {
            "name": "ProductAnalysis",
            "schema": {
                "type": "object",
                "properties": {
                    "product_name": {"type": "string"},
                    "price": {"type": "number", "minimum": 0},
                    "rating": {"type": "number", "minimum": 1, "maximum": 5},
                    "category": {
                        "type": "string", 
                        "enum": ["electronics", "clothing", "books", "home"]
                    },
                    "pros": {"type": "array", "items": {"type": "string"}},
                    "cons": {"type": "array", "items": {"type": "string"}},
                    "recommendation": {
                        "type": "string",
                        "enum": ["highly_recommended", "recommended", "neutral", "not_recommended"]
                    }
                },
                "required": ["product_name", "price", "rating", "category", "recommendation"]
            }
        }
    ],
    "agent": {
        "name": "product_analyzer",
        "model": "us.anthropic.claude-3-7-sonnet-20250219-v1:0",
        "system_prompt": "Analyze product information and provide structured insights.",
        "structured_output": {
            "schema": "ProductAnalysis",
            "validation": {
                "strict": True,
                "allow_extra_fields": False
            },
            "error_handling": {
                "retry_on_validation_error": True,
                "max_retries": 2
            }
        }
    }
}

# Load advanced agent with new loader instance
advanced_loader = AgentConfigLoader()
advanced_agent = advanced_loader.load_agent(advanced_config)
print("✅ Loaded advanced agent with validation")

In [ ]:
# Demo product analysis
product_review = """
I recently bought the Sony WH-1000XM4 headphones for $299. 
These are wireless noise-canceling headphones that deliver exceptional sound quality.

Pros:
- Excellent noise cancellation
- Great battery life (30+ hours)
- Comfortable for long listening sessions
- Quick charge feature

Cons:
- Expensive compared to competitors
- Touch controls can be finicky
- Bulky design

Overall, I'd rate these 4.5/5 stars. Highly recommended for audiophiles and frequent travelers.
"""

print("📄 Product Review:")
print(product_review[:200] + "...")

print("\n🔍 Analyzing product...")
analysis = advanced_agent.structured_output(f"Analyze this product review: {product_review}")

print("\n✅ Analysis completed!")
print(f"Product: {analysis.product_name}")
print(f"Price: ${analysis.price}")
print(f"Rating: {analysis.rating}/5")
print(f"Category: {analysis.category}")
print(f"Recommendation: {analysis.recommendation}")
print(f"Pros: {', '.join(analysis.pros)}")
print(f"Cons: {', '.join(analysis.cons)}")

# Show full structured data
display(JSON(analysis.model_dump(), expanded=True))

## Shared Schema Registry

You can also share schemas across multiple agents by defining them in a single configuration:

In [ ]:
# Configuration with shared schemas
shared_config = {
    "schemas": [
        {
            "name": "ContactInfo",
            "schema": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "email": {"type": "string"},
                    "phone": {"type": "string"},
                    "company": {"type": "string"}
                },
                "required": ["name", "email"]
            }
        },
        {
            "name": "TaskInfo",
            "schema": {
                "type": "object",
                "properties": {
                    "title": {"type": "string"},
                    "priority": {"type": "string", "enum": ["low", "medium", "high", "urgent"]},
                    "due_date": {"type": "string"},
                    "assignee": {"type": "string"}
                },
                "required": ["title", "priority"]
            }
        }
    ]
}

# Create multiple agents using shared schemas
shared_loader = AgentConfigLoader()

# Load schemas first
shared_loader._load_global_schemas(shared_config["schemas"])

# Create contact extractor agent
contact_agent_config = {
    "agent": {
        "name": "contact_extractor",
        "model": "us.anthropic.claude-3-7-sonnet-20250219-v1:0",
        "system_prompt": "Extract contact information from text.",
        "structured_output": "ContactInfo"
    }
}

# Create task extractor agent
task_agent_config = {
    "agent": {
        "name": "task_extractor",
        "model": "us.anthropic.claude-3-7-sonnet-20250219-v1:0",
        "system_prompt": "Extract task information from text.",
        "structured_output": "TaskInfo"
    }
}

contact_agent = shared_loader.load_agent(contact_agent_config)
task_agent = shared_loader.load_agent(task_agent_config)

print("✅ Created multiple agents with shared schemas")
print(f"📋 Available schemas: {list(shared_loader.schema_registry.list_schemas().keys())}")
print(f"👤 Contact agent schema: {contact_agent._structured_output_schema.__name__}")
print(f"📋 Task agent schema: {task_agent._structured_output_schema.__name__}")

## Summary

In this tutorial, we've learned how to:

✅ **Configure structured output schemas** using JSON Schema syntax in agent configurations

✅ **Extract structured data** from unstructured text using configured agents

✅ **Apply validation and constraints** to ensure data quality and consistency

✅ **Handle complex nested data structures** with multiple fields and validation rules

✅ **Share schemas across multiple agents** using a shared schema registry

## Key Patterns

### Single Agent with Schema
```python
config = {
    "schemas": [{"name": "MySchema", "schema": {...}}],
    "structured_output": "MySchema"
}
agent = AgentConfigLoader().load_agent(config)
```

### Multiple Agents with Shared Schemas
```python
loader = AgentConfigLoader()
loader._load_global_schemas(shared_schemas)
agent1 = loader.load_agent({"structured_output": "Schema1"})
agent2 = loader.load_agent({"structured_output": "Schema2"})
```

### Advanced Validation
```python
config = {
    "structured_output": {
        "schema": "MySchema",
        "validation": {"strict": True},
        "error_handling": {"max_retries": 3}
    }
}
```

## Next Steps

- Explore external schema files for complex, reusable schemas
- Build multi-agent systems with shared schema registries
- Integrate with databases and external APIs using structured output
- Deploy structured output agents in production environments

## Additional Resources

- [Structured Output Documentation](../../../sdk-python/src/strands/experimental/config_loader/agent/STRUCTURED-OUTPUT.md)
- [Pydantic Documentation](https://docs.pydantic.dev/)
- [JSON Schema Specification](https://json-schema.org/)